# Bài 5
Đây là notebook chứa mã nguồn đầy đủ của bài 5.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
PROJECTS = [
    # id, name, cost, npv, cost_y12, cost_y35, gdp_imp, equity, ai_ready
    (1, 'TT Dữ liệu Hòa Lạc', 12000, 21500, 8500, 3500, 15, 2, 8),
    (2, 'TT Dữ liệu phía Nam', 11500, 20800, 7500, 4000, 14, 3, 7),
    (3, '5G toàn quốc', 18000, 32500, 12000, 6000, 20, 10, 5),
    (4, 'VNeID 2.0', 4500, 9200, 3500, 1000, 5, 8, 2),
    (5, 'Dịch vụ công v3', 3200, 6800, 2500, 700, 4, 9, 2),
    (6, 'Y tế số', 5800, 11400, 4000, 1800, 6, 12, 4),
    (7, 'Giáo dục số', 6500, 12200, 4500, 2000, 5, 15, 5),
    (8, 'TT AI quốc gia', 15000, 28500, 9000, 6000, 18, 2, 25),
    (9, 'Sandbox Fintech', 2500, 5800, 1800, 700, 6, 4, 3),
    (10, 'Logistics số', 7200, 13800, 5000, 2200, 10, 3, 6),
    (11, 'Nông nghiệp số', 4800, 8500, 3500, 1300, 5, 14, 3),
    (12, 'Đào tạo 50k kỹ sư', 8500, 16200, 5500, 3000, 8, 10, 15),
    (13, 'KCN Bán dẫn', 20000, 35000, 13000, 7000, 25, 2, 10),
    (14, 'An ninh mạng SOC', 3800, 7500, 2800, 1000, 3, 3, 8),
    (15, 'Open Data', 1500, 3800, 1200, 300, 2, 8, 4),
]

In [ ]:
def solve_bai05(budget=80000, w_gdp=0.40, w_equity=0.30, w_ai=0.30):
    P = [p[0] for p in PROJECTS]
    C = {p[0]: p[2] for p in PROJECTS}
    B = {p[0]: p[3] for p in PROJECTS}
    C12 = {p[0]: p[4] for p in PROJECTS}
    gdp_imp = {p[0]: p[6] for p in PROJECTS}
    equity = {p[0]: p[7] for p in PROJECTS}
    ai_ready = {p[0]: p[8] for p in PROJECTS}
    # risk probs
    probs = {1:0.85, 2:0.85, 3:0.90, 4:0.75, 5:0.75, 6:0.80, 7:0.80, 8:0.65, 9:0.60, 10:0.75, 11:0.85, 12:0.95, 13:0.70, 14:0.90, 15:0.95}

    def solve_pulp(B_cap, force_p1p2=False, use_risk=False):
        m = pulp.LpProblem('VN_Project_Selection', pulp.LpMaximize)
        y = pulp.LpVariable.dicts('y', P, cat='Binary')
        
        obj = []
        for i in P:
            adjusted_b = B[i] * (1.0 + w_gdp*gdp_imp[i]/20.0 + w_equity*equity[i]/10.0 + w_ai*ai_ready[i]/15.0)
            if use_risk:
                adjusted_b *= probs[i]
            obj.append(adjusted_b * y[i])
            
        m += pulp.lpSum(obj)
        m += pulp.lpSum(C[i]*y[i] for i in P) <= B_cap
        m += pulp.lpSum(C12[i]*y[i] for i in P) <= B_cap / 2.0
        
        if not force_p1p2:
            m += y[1] + y[2] <= 1
        else:
            m += y[1] + y[2] == 2
            
        m += y[8] <= y[12]
        m += y[13] <= y[12]
        m += y[4] + y[5] >= 1
        m += y[14] >= 1
        m += pulp.lpSum(y[i] for i in P) >= 7
        m += pulp.lpSum(y[i] for i in P) <= 11
        
        m.solve(pulp.PULP_CBC_CMD(msg=False))
        ok = m.status == pulp.LpStatusOptimal
        if ok:
            selected = [p[1] for p in PROJECTS if pulp.value(y[p[0]]) > 0.5]
            total_cost = sum(C[i] for i in P if pulp.value(y[i]) > 0.5)
            total_val = sum(B[i] for i in P if pulp.value(y[i]) > 0.5)
            z = pulp.value(m.objective)
            return True, z, total_cost, total_val, selected
        return False, 0, 0, 0, []

    # 1. Base solve
    ok_base, z_base, cost_base, npv_base, sel_base = solve_pulp(budget)
    project_details = {}
    if ok_base:
        for p in PROJECTS:
            if p[1] in sel_base:
                project_details[p[1]] = {
                    'cost': C[p[0]], 'npv': B[p[0]], 'gdp_impact': gdp_imp[p[0]]*1000,
                    'equity': equity[p[0]]*1000, 'ai_readiness': ai_ready[p[0]]*1000
                }
                
    # 2. Budget = 100k
    ok_100k, z_100k, _, _, sel_100k = solve_pulp(100000)
    res_100k = {'status': 'Optimal' if ok_100k else 'Infeasible', 'Z': z_100k, 'selected': sel_100k}
    
    # 3. Force P1+P2
    ok_p, z_p, _, _, sel_p = solve_pulp(budget, force_p1p2=True)
    res_p1p2 = {'status': 'Optimal' if ok_p else 'Infeasible', 'Z': z_p, 'selected': sel_p}
    
    # 4. Risk
    ok_r, z_r, _, _, sel_r = solve_pulp(budget, use_risk=True)
    res_risk = {'status': 'Optimal' if ok_r else 'Infeasible', 'Z': z_r, 'selected': sel_r}
    prob_completion = {p[1]: probs[p[0]] for p in PROJECTS}
    
    return {
        'status': 'Optimal' if ok_base else 'Infeasible',
        'Z': z_base,
        'cost': cost_base,
        'total_npv': npv_base,
        'npv_margin': z_base / cost_base if cost_base > 0 else 0,
        'selected': sel_base,
        'projects': project_details,
        'res_100k': res_100k,
        'res_p1p2': res_p1p2,
        'res_risk': res_risk,
        'prob_completion': prob_completion
    }

In [ ]:
if __name__ == '__main__':
    res = solve_bai05()
    # In ra một số key để kiểm tra
    if isinstance(res, dict):
        print(res.keys())